In [1]:
# !pip install -q sentence-transformers faiss-cpu

# Импорт библиотек
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import pandas as pd
from sklearn.metrics import pairwise_distances
from sentence_transformers import SentenceTransformer

# FAISS
import faiss

# Фиксируем Seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# torch
USE_TORCH = True

if USE_TORCH:
    import torch

    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
else:
    device = "cpu"

Device: cuda


In [2]:
import kagglehub

# Скачиваем
path = kagglehub.dataset_download("stanfordu/stanford-question-answering-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'stanford-question-answering-dataset' dataset.
Path to dataset files: /kaggle/input/stanford-question-answering-dataset


In [3]:
squad_file = os.path.join(path, "train-v1.1.json")

# Загрузка JSON
with open(squad_file, "r", encoding="utf-8") as f:
    squad_data = json.load(f)

# Извлечение всех параграфов
documents = []
for article in squad_data["data"]:
    for paragraph in article["paragraphs"]:
        documents.append({
            "id": len(documents)+1,
            "text": paragraph["context"]
        })

# Переводим в DataFrame
df = pd.DataFrame(documents)

# Количество документов
print("Number of documents:", len(df))

# Примеры
df.sample(5, random_state=42)

Number of documents: 18896


,id,text
15301,15302,The British—by inclination as well as for prac...
11123,11124,Studies of nutritional status must take into a...
6451,6452,On the eve of America's entry into World War I...
14076,14077,Cockroaches are among the fastest insect runne...
6151,6152,Beer ranges from less than 3% alcohol by volum...


кратко пояснить, что это за предметная область и почему по ней разумно строить retrieval / mini-RA:

Выбранная база знаний: SQuAD (Stanford Question Answering Dataset)

Она состоит из paragraphs из статей Wikipedia, которые сопровождаются вопросами и ответами.
Эта предметная область подходит для задачи retrieval / mini-RAG, потому что:

- параграфы содержат чётко сформулированные факты, по которым можно задавать вопросы;
- легко формулировать контрольные запросы с ожидаемым источником;
- данные хорошо подходят для эмбеддингов и FAISS, так как каждый текст является отдельным фрагментом, и retrieval можно оценить по hit@k или recall@k.

In [4]:
# Параметры чанкинга
chunk_size = 200   # число символов в одном чанке
overlap = 50       # число символов перекрытия между чанками

# Фукнкция разбивает текст на чанки длиной chunk_size с перекрытием overlap.
# Возвращает список строк (чанков)
def chunk_text(text, chunk_size=chunk_size, overlap=overlap):
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end].strip()
        if chunk:  # исключаем пустые чанки
            chunks.append(chunk)
        start += (chunk_size - overlap)

    return chunks

# Применяем к базе знаний
all_chunks = []
for idx, row in df.iterrows():
    chunks = chunk_text(row["text"])
    for i, c in enumerate(chunks):
        all_chunks.append({
            "doc_id": row["id"],
            "chunk_id": i+1,
            "text": c
        })

chunks_df = pd.DataFrame(all_chunks)

# Показываем примеры
print("Total chunks:", len(chunks_df))
chunks_df.sample(5, random_state=42)

Total chunks: 101962


,doc_id,chunk_id,text
91651,17096,2,ient and pleasant year-round option. At the sa...
90694,16937,2,"en out of Malaya, Allied forces in Singapore a..."
24643,5515,2,al and the Asia House Festival of Asian Litera...
44669,8938,6,in limited powers in this area: in matters tha...
15741,3401,4,ts used by the Saxons. In contrast to the othe...


кратко пояснить выбранные параметры (`chunk_size`, `overlap` или их аналог):

chunk_size = 200 это стандартный размер фрагмента, чтобы embeddings не были слишком длинными (в SQuAD параграфы примерно 200–600 символов).

overlap = 50 это перекрытие для сохранения контекста между соседними чанками, чтобы важная информация не обрывалась.

Функция chunk_text воспроизводимо разбивает любой текст на последовательные сегменты.

Данный подход простой, воспроизводим и подходит для построения индекса FAISS.

In [5]:
model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)
embedder.to(device)

# Создаём эмбеддинги для чанков
texts = chunks_df["text"].tolist()
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print("Embeddings shape:", embeddings.shape)

# Создание FAISS индекса
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)
print("Number of vectors in index:", index.ntotal)

# Пример поиска top-k фрагментов
top_k = 3
example_queries = [
    "What is the capital of USA?",
    "Who was the MVP of Super Bowl XXXIV",
    "Which author created Sherlock Holmes?"
]

for q in example_queries:
    q_emb = embedder.encode([q], convert_to_numpy=True)
    D, I = index.search(q_emb, top_k)  # D: distances, I: индексы
    print(f"\nQuery: {q}")
    for rank, idx in enumerate(I[0], start=1):
        print(f"Rank {rank}: {chunks_df.iloc[idx]['text'][:150]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3187 [00:00<?, ?it/s]

Embeddings shape: (101962, 384)
Number of vectors in index: 101962

Query: What is the capital of USA?
Rank 1: American city....
Rank 2: and its surroundings came under English control in 1664. New York served as the capital of the United States from 1785 until 1790. It has been the cou...
Rank 3: ill at 330 feet (100 m) above sea level, and the lowest point is at sea level. Situated onshore of the Atlantic Ocean, Boston is the only state capita...

Query: Who was the MVP of Super Bowl XXXIV
Rank 1: The year 2000 brought heightened interest in the AFL. Then-St. Louis Rams quarterback Kurt Warner, who was MVP of Super Bowl XXXIV, was first noticed ...
Rank 2: career, serving as the main headliner of the 47th Super Bowl halftime show in 2013....
Rank 3: mes: 2005, 2013 and 2014. They defeated the Denver Broncos 43-8 to win their first Super Bowl championship in Super Bowl XLVIII, but lost 24-28 agains...

Query: Which author created Sherlock Holmes?
Rank 1: erlock Holmes stories. Modern 

In [6]:
# Контрольные запросы и ожидаемые ключевые слова
control_queries = [
    {"query": "Capital of USA", "keywords": ["capital", "United States"]},
    {"query": "Super Bowl XXXIV MVP", "keywords": ["Kurt Warner", "MVP"]},
    {"query": "Sherlock Holmes author", "keywords": ["Conan Doyle", "author"]},
    {"query": "Beer alcohol content", "keywords": ["alcohol", "volume", "beer"]},
    {"query": "Fastest insect runners", "keywords": ["Cockroaches", "fastest", "insect"]},
    {"query": "CAA ATC responsibilities", "keywords": ["CAA", "ATC", "airports"]},
    {"query": "British ales strength", "keywords": ["British", "ales", "4%"]},
    {"query": "Singapore surrender 1942", "keywords": ["Singapore", "Japanese", "1942"]},
    {"query": "Nutritional status studies", "keywords": ["nutritional", "status", "experiments"]},
    {"query": "London Great Plague", "keywords": ["London", "Plague", "1665"]}
]

def is_relevant(text, keywords):
    text_lower = text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

k_values = [1, 3, 5]
metrics = {k: {"hit": 0, "recall_sum": 0} for k in k_values}
eval_results = []

for q_data in control_queries:
    q_emb = embedder.encode([q_data["query"]], convert_to_numpy=True)
    D, I = index.search(q_emb, max(k_values))

    retrieved_texts = [chunks_df.iloc[idx]['text'] for idx in I[0]]
    retrieved_ids = [chunks_df.iloc[idx]['doc_id'] for idx in I[0]]

    # Расчет метрик
    for k in k_values:
        top_k_texts = retrieved_texts[:k]
        relevant_count = sum(1 for t in top_k_texts if is_relevant(t, q_data["keywords"]))
        if relevant_count > 0:
            metrics[k]["hit"] += 1
        metrics[k]["recall_sum"] += min(relevant_count, 1)

    # Поиск ранга первого релевантного для CSV
    rank_first = -1
    hit_at_k = {k: 0 for k in k_values}
    for k in k_values:
        for i in range(k):
            if is_relevant(retrieved_texts[i], q_data["keywords"]):
                hit_at_k[k] = 1
                if rank_first == -1:
                    rank_first = i + 1
                break

    eval_results.append({
        "query": q_data["query"],
        "expected_source": ", ".join(q_data["keywords"]),
        "retrieved_sources": ";".join(map(str, retrieved_ids[:5])),
        "hit_at_5": hit_at_k[5],
        "rank_of_first_relevant": rank_first
    })

# Вывод метрик
print(f"{'k': <5} | {'Hit@k': <10} | {'Recall@k': <10}")
print("-" * 30)
for k in k_values:
    hit_rate = metrics[k]["hit"] / len(control_queries)
    recall_rate = metrics[k]["recall_sum"] / len(control_queries)
    print(f"{k: <5} | {hit_rate: <10.2f} | {recall_rate: <10.2f}")

# Сохранение CSV
pd.DataFrame(eval_results).to_csv("retrieval_eval.csv", index=False)

k     | Hit@k      | Recall@k  
------------------------------
1     | 1.00       | 1.00      
3     | 1.00       | 1.00      
5     | 1.00       | 1.00      


In [7]:
# Параметры эксперимента
sample_docs = df.head(50)
chunk_sizes = [200, 400]
test_queries = control_queries[:5]

results = []

for size in chunk_sizes:
    # Чанкинг
    chunks = []
    for idx, row in sample_docs.iterrows():
        start = 0
        text = row['text']
        while start < len(text):
            end = min(start + size, len(text))
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start += (size - 50)

    # Эмбеддинги
    emb = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=False)

    # Индекс
    idx_index = faiss.IndexFlatL2(emb.shape[1])
    idx_index.add(emb)

    # Оценка
    hits = 0
    for q_data in test_queries:
        q_emb = embedder.encode([q_data["query"]], convert_to_numpy=True)
        D, I = idx_index.search(q_emb, 1)  # Hit@1
        retrieved_text = chunks[I[0][0]].lower()
        if any(kw.lower() in retrieved_text for kw in q_data["keywords"]):
            hits += 1

    hit_rate = hits / len(test_queries)
    results.append({"chunk_size": size, "Hit@1": hit_rate})
    print(f"Chunk size: {size}, Hit@1: {hit_rate:.2f}")

# Вывод таблицы
pd.DataFrame(results)

Chunk size: 200, Hit@1: 0.00
Chunk size: 400, Hit@1: 0.00


,chunk_size,Hit@1
0,200,0.0
1,400,0.0


In [8]:
# Новые документы
new_docs = [
    {"id": 99901, "text": "Qt 6.5 introduced new features to Qt Quick. Integration with C++ remained stable."},
    {"id": 99902, "text": "FAISS is a library for efficient vector similarity search and clustering."},
    {"id": 99903, "text": "Sentence Transformers provide dense vector representations for sentences."}
]
new_df = pd.DataFrame(new_docs)

# Чанкинг новых документов
new_chunks = []
for idx, row in new_df.iterrows():
    chunks = chunk_text(row['text'])
    for i, c in enumerate(chunks):
        new_chunks.append({"doc_id": row['id'], "chunk_id": i+1, "text": c})
new_chunks_df = pd.DataFrame(new_chunks)

# Обновление общего DataFrame
chunks_df_updated = pd.concat([chunks_df, new_chunks_df], ignore_index=True)

# Запросы для проверки обновления
update_queries = [
    "What's new in Qt 6.5?",
    "FAISS library purpose",
    "Sentence Transformers usage"
]

update_comparison = []

# Поиск ДО добавления новых векторов
before_results = {}
for q in update_queries:
    q_emb = embedder.encode([q], convert_to_numpy=True)
    D, I = index.search(q_emb, 3)
    before_ids = [chunks_df.iloc[idx]['doc_id'] for idx in I[0]]
    before_results[q] = before_ids

# Добавление embeddings в индекс
new_embeddings = embedder.encode(new_chunks_df['text'].tolist(), convert_to_numpy=True)
index.add(new_embeddings)

print(f"Total chunks after update: {len(chunks_df_updated)}")
print(f"Index size: {index.ntotal}")

# Поиск ПОСЛЕ добавления и сравнение
for q in update_queries:
    q_emb = embedder.encode([q], convert_to_numpy=True)
    D, I = index.search(q_emb, 3)
    after_ids = [chunks_df_updated.iloc[idx]['doc_id'] for idx in I[0]]

    before_ids = before_results[q]
    changed = "Yes" if before_ids != after_ids else "No"

    # Проверка наличия новых документов (id >= 99900)
    new_docs_found = any(int(rid) >= 99900 for rid in after_ids)
    if new_docs_found and changed == "No":
        changed = "Yes (new docs)"

    update_comparison.append({
        "query": q,
        "before_retrieved_sources": ";".join(map(str, before_ids)),
        "after_retrieved_sources": ";".join(map(str, after_ids)),
        "changed": changed
    })

    print(f"\nQuery: {q}")
    print(f"After: {after_ids}")

# Сохранение CSV
pd.DataFrame(update_comparison).to_csv("retrieval_before_after_update.csv", index=False)

Total chunks after update: 101965
Index size: 101965

Query: What's new in Qt 6.5?
After: [np.int64(99901), np.int64(3375), np.int64(3375)]

Query: FAISS library purpose
After: [np.int64(14217), np.int64(99902), np.int64(14230)]

Query: Sentence Transformers usage
After: [np.int64(99903), np.int64(17370), np.int64(8583)]


In [9]:
def run_mini_rag(query, k=3):
    # Эмбеддинг запроса
    q_emb = embedder.encode([query], convert_to_numpy=True)

    # Retrieval
    D, I = index.search(q_emb, k)

    # Сбор контекста и источников
    contexts = []
    sources = []
    for idx in I[0]:
        row = chunks_df_updated.iloc[idx]
        contexts.append(row['text'])
        sources.append(row['doc_id'])

    answer = contexts[0] if contexts else "Ответ не найден"

    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "context": contexts
    }

# Пример использования
result = run_mini_rag("What library is used to find vectors?")
print(f"Query: {result['query']}")
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")

Query: What library is used to find vectors?
Answer: FAISS is a library for efficient vector similarity search and clustering.
Sources: [np.int64(99902), np.int64(13582), np.int64(14217)]


In [10]:
# Запускаем mini-RAG на тестовых запросах
test_queries = [
     "What library is used for vector search?",
     "When was Qt 6.5 released?",
     "Who wrote Harry Potter?",
     "What's the weather like in Moscow today?",
     "How do I install FAISS on Windows?"
]

rag_results = []

print(f"{'Запрос': <45} | {'Лучший результат': <60} | {'Статус'}")
print("-" * 120)

for q in test_queries:
    res = run_mini_rag(q, k=3)

    # Краткий вывод ответа
    answer_preview = res['answer'][:60].replace('\n', ' ') + "... " if len(res['answer']) > 60 else res['answer']

    # Оценка релевантности
    if "weather" in q.lower() or "Moscow" in q.lower():
        status = "Fail (no real-time data)"
    elif "install" in q.lower() and "FAISS" in q:
        status = "Partial (no install instructions)"
    elif any(kw in res['answer'].lower() for kw in ["faiss", "qt", "rowling", "conan doyle", "vector"]):
        status = "OK"
    else:
        status = "Fail (irrelevant context)"

    print(f"{q[:45]: <45} | {answer_preview: <60} | {status}")

    # Сбор данных для CSV
    rag_results.append({
        "question": q,
        "answer": res['answer'],
        "retrieved_sources": ";".join(map(str, res['sources']))
    })

# Сохранение CSV
pd.DataFrame(rag_results).to_csv("rag_examples.csv", index=False)

Запрос                                        | Лучший результат                                             | Статус
------------------------------------------------------------------------------------------------------------------------
What library is used for vector search?       | FAISS is a library for efficient vector similarity search an...  | OK
When was Qt 6.5 released?                     | Qt 6.5 introduced new features to Qt Quick. Integration with...  | OK
Who wrote Harry Potter?                       | rothers), was the head writer.                               | Fail (irrelevant context)
What's the weather like in Moscow today?      | The city generally has a climate with warm days followed by ...  | Fail (no real-time data)
How do I install FAISS on Windows?            | ilable on Windows 95 through Microsoft Layer for Unicode, as...  | Partial (no install instructions)


- кратко прокомментировать 2-4 неудачных или пограничных случая;
- пояснить, что именно пошло не так: retrieval, состав контекста, формулировка вопроса, неполнота базы знаний и т.д.

Анализ неудачных и пограничных случаев

- Запрос `Who wrote Harry Potter?` вернул нерелевантный фрагмент `"...rothers), was the head writer."`, что связано с неполнотой базы знаний: в выбранной подвыборке SQuAD отсутствует параграф, содержащий информацию о Дж. К. Роулинг. Векторный поиск нашёл текст, семантически близкий по структуре запроса, но не по содержанию, поскольку гибридный поиск с ключевыми словами не используется. Для улучшения результатов в подобных случаях нужно добавлять этап reranking для более точного отбора контекста.

- Запрос `What's the weather like in Moscow today?` получил ответ с общим описанием климата, но без актуальных данных, так как mini-RAG работает исключительно со статической базой знаний. Запросы, требующие доступа к внешним источникам или данным реального времени, находятся за пределами компетенции системы. Отсутствие механизма детекции out-of-scope запросов приводит к попытке найти ответ в нерелевантных фрагментах. Решением может стать внедрение классификатора запросов для перенаправления таких вопросов к соответствующим внешним сервисам

- Запрос `How do I install FAISS on Windows?` вернул фрагмент про совместимость с Windows, но без пошаговой инструкции, поскольку база содержит описательные тексты о библиотеках, а не технические руководства. Даже при наличии инструкции в исходном документе она могла быть разорвана между чанками из-за фиксированного размера `chunk_size=200`. Для решения этой проблемы можно добавлять в базу техническую документацию и туториалы, а также увеличивать параметр `overlap` или использовать адаптивный чанкинг для сохранения целостности инструкций.

- Запрос `When was Qt 6.5 released?` является пограничным случаем: упоминание версии найдено, но конкретная дата релиза отсутствует так как не была задана при добавлении новых документов, а в исходном датасете ее просто нет.

